# 05b - Modelo 2: Clasificación de Tipo de Anomalía con LightGBM
 
 Variante paralela del cuaderno 05. Usa las predicciones/features del detector LightGBM y entrena `LGBMClassifier` para clasificar el tipo de anomalía.
 
 **Entrada:** `data/interim/04b_modelo1_predicciones_lightgbm_full_v3.parquet` + artefactos LightGBM del paso anterior  
 **Salida:** `data/models/modelo_2_clasificador_lightgbm_full_v3.joblib`


In [ ]:
# ─── Intel Extension for Scikit-learn ────────────────────────────────────────
# IMPORTANTE: debe cargarse ANTES que cualquier import de sklearn.
# Instalar con: pip install scikit-learn-intelex
# En CPU Intel puede acelerar Random Forest e IterativeImputer 10-100x.
try:
    from sklearnex import patch_sklearn
    patch_sklearn()
    _intel_activo = True
except ImportError:
    _intel_activo = False

# ─── Librerías estándar ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os, glob, joblib, pickle, warnings
import matplotlib.ticker as mticker

# ─── Scikit-learn (ya parcheado si Intel Extension está disponible) ───────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.experimental import enable_iterative_imputer  # requerido antes de IterativeImputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              classification_report, confusion_matrix,
                              average_precision_score)
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

# ─── Configuración del proyecto ───────────────────────────────────────────────
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from config import *

# Rutas aisladas experimento FULL_V3
EXPERIMENTO_FULL = 'full_v3'
PARQUET_03 = os.path.join(DATA_INTERIM, '03_datos_features_full_v3.parquet')
PARQUET_04_LGBM = os.path.join(DATA_INTERIM, '04b_modelo1_predicciones_lightgbm_full_v3.parquet')
MODELO_2_LGBM_PATH = os.path.join(DATA_MODELS, 'modelo_2_clasificador_lightgbm_full_v3.joblib')
LABEL_ENC_M2_LGBM_PATH = os.path.join(DATA_MODELS, 'label_encoder_modelo_2_lightgbm_full_v3.joblib')
IMPUTER_M1_LGBM_PATH = os.path.join(DATA_MODELS, 'imputer_modelo_1_lightgbm_full_v3.joblib')
FEATURES_M1_LGBM_PATH = os.path.join(DATA_MODELS, 'features_modelo_1_lightgbm_full_v3.joblib')
BASELINES_CTX_LGBM_PATH = os.path.join(DATA_MODELS, 'baselines_ctx_lightgbm_full_v3.joblib')
print('Experimento:', EXPERIMENTO_FULL)
print('Features:', PARQUET_03)
print('Predicciones M1:', PARQUET_04_LGBM)

if _intel_activo:
    print("Intel Extension for Scikit-learn ACTIVA — algoritmos acelerados con oneAPI")
else:
    print("Tip: instala scikit-learn-intelex para acelerar en CPU Intel")
    print("     pip install scikit-learn-intelex")


## 0. Cargar datos y modelos del paso anterior

In [ ]:
# Cargar dataset de features (PARQUET_03) — contiene todas las features incluyendo *_rstd_5m.
# PARQUET_04 no preserva las features *_rstd_vs_expected que el imputer espera.
df_trabajo = pd.read_parquet(PARQUET_03)
print(f"Dataset cargado: {df_trabajo.shape}")

if COLUMNA_USAR_EN_TRAIN in df_trabajo.columns:
    resumen_uso = df_trabajo[COLUMNA_USAR_EN_TRAIN].value_counts(dropna=False)
    print(f"Distribucion de '{COLUMNA_USAR_EN_TRAIN}':")
    print(resumen_uso)
    filas_totales = len(df_trabajo)
    df_trabajo = df_trabajo.loc[df_trabajo[COLUMNA_USAR_EN_TRAIN].fillna(False).astype(bool)].reset_index(drop=True)
    print(f"Filas usadas Modelo 2: {len(df_trabajo):,} / {filas_totales:,}")
else:
    print(f"Columna '{COLUMNA_USAR_EN_TRAIN}' no disponible: PARQUET_03 ya viene filtrado por train; se usan todas sus filas.")

# Cargar el imputer entrenado en 04, la lista de features de salida y los baselines contextuales
imputer      = joblib.load(IMPUTER_M1_LGBM_PATH)
features_out = joblib.load(FEATURES_M1_LGBM_PATH)
ctx_data     = joblib.load(BASELINES_CTX_LGBM_PATH)
stats_ctx    = ctx_data['stats']
sensores_ctx = ctx_data['sensores']

# Reproducir las features *_rstd_vs_expected usando los baselines guardados del fold final de 04
merged = df_trabajo[['Hora', 'Mes']].merge(stats_ctx, on=['Hora', 'Mes'], how='left')
merged.index = df_trabajo.index
for col in sensores_ctx:
    std_c  = f'{col}_std_hora_ctx'
    rstd_c = f'{col}_rstd_5m'
    if std_c in merged.columns and rstd_c in df_trabajo.columns:
        df_trabajo[f'{col}_rstd_vs_expected'] = (
            df_trabajo[rstd_c] / (merged[std_c].fillna(1).values + 1e-6)
        )

X = df_trabajo[imputer.feature_names_in_].copy()
y = df_trabajo['etiqueta_deteccion'].copy()

# Reproducir el mismo split que en 04
tscv = TimeSeriesSplit(n_splits=TSCV_N_SPLITS)
splits = list(tscv.split(X))
train_idx, test_idx = splits[-1]   # último fold (mismo que en 04)
X_train = X.iloc[train_idx];  X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx];  y_test = y.iloc[test_idx]

# IMPORTANTE: preservar el índice original para que y_train/y_test y X_*_imputed estén alineados
X_train_imputed = pd.DataFrame(
    imputer.transform(X_train),
    columns=features_out,
    index=X_train.index
)
X_test_imputed = pd.DataFrame(
    imputer.transform(X_test),
    columns=features_out,
    index=X_test.index
)
print(f"Features input: {len(imputer.feature_names_in_)}  →  output tras imputación: {len(features_out)}")
print("Imputación cargada y aplicada.")

## 1. Preparar datos de entrenamiento para Modelo 2
Filtramos solo las filas que **realmente son anomalías** en el conjunto de entrenamiento. El Modelo 2 solo ve anomalías — aprende a distinguir entre tipos, no entre normal y anomalía.

In [ ]:
if ('X_train_imputed' in locals() and 'y_train' in locals() and
    'X_test_imputed' in locals() and 'y_test' in locals() and
    'df_trabajo' in locals()):

    # --- Preparar datos para Modelo 2 ---
    # full_v3: estas clases se resuelven por reglas deterministas, no por el
    # clasificador multiclase. Se conservan para deteccion/correccion, pero
    # quedan excluidas del entrenamiento/evaluacion del Modelo 2.
    # - Valores Fuera de Rango: violacion directa de RANGOS_FISICOS.
    # - Datos Faltantes: NaN real/sintetico atribuido al sensor concreto.
    CLASES_DETERMINISTICAS_MODELO2 = ['Valores Fuera de Rango', 'Datos Faltantes']

    # ENTRENAMIENTO: anomalias verdaderas no deterministas.
    indices_anomalias_train_all = X_train_imputed[y_train == 'anomalia'].index
    y_train_tipo_all = df_trabajo.loc[indices_anomalias_train_all, 'etiqueta_tipo_anomalia']
    mask_train_m2 = ~y_train_tipo_all.isin(CLASES_DETERMINISTICAS_MODELO2)
    indices_anomalias_train = indices_anomalias_train_all[mask_train_m2]
    X_model2_train = X_train_imputed.loc[indices_anomalias_train]
    y_model2_train_str = df_trabajo.loc[indices_anomalias_train, 'etiqueta_tipo_anomalia']

    print(f"Excluidas del Modelo 2 train por regla determinista: {int((~mask_train_m2).sum()):,}")
    print(f"Forma de X_model2_train: {X_model2_train.shape}")
    if X_model2_train.empty:
        print("Advertencia: No se encontraron anomal?as no deterministas en el conjunto de entrenamiento para Modelo 2.")
    else:
        print("Distribuci?n de tipos de anomal?a en y_model2_train_str:")
        print(y_model2_train_str.value_counts())

    # PRUEBA: anomalias verdaderas no deterministas.
    indices_anomalias_test_all = X_test_imputed[y_test == 'anomalia'].index
    y_test_tipo_all = df_trabajo.loc[indices_anomalias_test_all, 'etiqueta_tipo_anomalia']
    mask_test_m2 = ~y_test_tipo_all.isin(CLASES_DETERMINISTICAS_MODELO2)
    indices_anomalias_test = indices_anomalias_test_all[mask_test_m2]
    X_model2_test = X_test_imputed.loc[indices_anomalias_test]
    y_model2_test_str = df_trabajo.loc[indices_anomalias_test, 'etiqueta_tipo_anomalia']

    print(f"\nExcluidas del Modelo 2 test por regla determinista: {int((~mask_test_m2).sum()):,}")
    print(f"Forma de X_model2_test: {X_model2_test.shape}")
    if X_model2_test.empty:
        print("Advertencia: No se encontraron anomal?as no deterministas en el conjunto de prueba para Modelo 2.")
    else:
        print("Distribuci?n de tipos de anomal?a en y_model2_test_str:")
        print(y_model2_test_str.value_counts())

    # --- Codificaci?n de Etiquetas de Tipo de Anomal?a ---
    if not X_model2_train.empty and not X_model2_test.empty:
        label_encoder_model2 = LabelEncoder()

        # Ajustar el codificador SOLO con las etiquetas de entrenamiento.
        y_model2_train_encoded = label_encoder_model2.fit_transform(y_model2_train_str)
        y_model2_test_encoded = label_encoder_model2.transform(y_model2_test_str)

        print("\nEtiquetas codificadas para Modelo 2:")
        print(f"Primeras 5 y_model2_train_encoded: {y_model2_train_encoded[:5]}")
        print(f"Primeras 5 y_model2_test_encoded: {y_model2_test_encoded[:5]}")

        clases_modelo2 = label_encoder_model2.classes_
        print(f"Clases del Modelo 2 (codificadas de 0 a {len(clases_modelo2)-1}): {clases_modelo2}")

        clases_deterministas_presentes = set(clases_modelo2) & set(CLASES_DETERMINISTICAS_MODELO2)
        if clases_deterministas_presentes:
            raise RuntimeError(
                f"Clases deterministas no deben estar en Modelo 2: {sorted(clases_deterministas_presentes)}"
            )

        if pd.Series(y_model2_train_encoded).isnull().sum() > 0 or pd.Series(y_model2_test_encoded).isnull().sum() > 0:
            print("?Advertencia! Se encontraron NaNs en las etiquetas codificadas del Modelo 2.")

    else:
        print("\nNo se proceder? con la codificaci?n de etiquetas ya que no hay suficientes datos de anomal?as no deterministas.")

else:
    print("Error: Datos necesarios para Modelo 2 no est?n definidos ('X_train_imputed', 'y_train', etc.).")


## 2. Entrenar LightGBM Clasificador de Tipos

In [ ]:
# Verificar que los datos para Modelo 2 existan y no estén vacíos
if ('X_model2_train' in locals() and not X_model2_train.empty and
    'y_model2_train_encoded' in locals() and len(y_model2_train_encoded) > 0):

    # Inicializar el clasificador LightGBM para Modelo 2
    # class_weight='balanced' es especialmente importante aquí, ya que el número de instancias de cada tipo de anomalía puede ser muy diferente.
    try:
        from lightgbm import LGBMClassifier
    except ImportError as e:
        raise ImportError('LightGBM no está instalado. Instala con: pip install lightgbm') from e
    rf_clasificador_tipo = LGBMClassifier(**MODELO_2_LGBM_PARAMS)

    print("\nEntrenando el Modelo 2 (Clasificador de Tipos de Anomalía)...")
    rf_clasificador_tipo.fit(X_model2_train, y_model2_train_encoded)
    print("Entrenamiento del Modelo 2 completado.")

else:
    print("Error: Datos de entrenamiento para Modelo 2 (X_model2_train, y_model2_train_encoded) no están definidos o están vacíos.")
    # Detener la ejecución de los siguientes bloques si no hay datos para entrenar
    rf_clasificador_tipo = None

## 3. Evaluar Modelo 2

In [ ]:
if (rf_clasificador_tipo is not None and
    'X_model2_test' in locals() and not X_model2_test.empty and
    'y_model2_test_encoded' in locals() and len(y_model2_test_encoded) > 0 and
    'clases_modelo2' in locals()):

    print("\nRealizando predicciones con Modelo 2 en el conjunto de prueba...")
    y_pred_model2 = rf_clasificador_tipo.predict(X_model2_test)
    # y_pred_proba_model2 = rf_clasificador_tipo.predict_proba(X_model2_test) # Para métricas basadas en probabilidad si se necesitaran

    print("\n--- Evaluación del Modelo 2 (Clasificador de Tipos de Anomalía) ---")
    
    # Accuracy General
    accuracy_model2 = accuracy_score(y_model2_test_encoded, y_pred_model2)
    print(f"Accuracy General del Modelo 2: {accuracy_model2:.4f}")

    # Matriz de Confusión Multi-clase
    print("\nMatriz de Confusión del Modelo 2:")
    cm_model2 = confusion_matrix(y_model2_test_encoded, y_pred_model2)
    # print(cm_model2) # Imprimir la matriz numérica si se desea

    # Visualizar la matriz de confusión
    plt.figure(figsize=(10, 8)) # Ajustar tamaño según número de clases
    sns.heatmap(cm_model2, annot=True, fmt='d', cmap='Greens',
                xticklabels=clases_modelo2, yticklabels=clases_modelo2)
    plt.xlabel('Predicción del Tipo de Anomalía')
    plt.ylabel('Tipo de Anomalía Real')
    plt.title('Matriz de Confusión - Modelo 2 (Clasificador de Tipos)')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    # Reporte de Clasificación Detallado (Precisión, Recall, F1-score por clase)
    print("\nReporte de Clasificación del Modelo 2:")
    print(classification_report(y_model2_test_encoded, y_pred_model2, target_names=clases_modelo2))

else:
    print("Error: Modelo 2 no entrenado o datos de prueba para Modelo 2 no disponibles/vacíos.")
    if 'clases_modelo2' not in locals():
        print("  Además, 'clases_modelo2' (nombres de las clases) no está definido.")

## 4. Guardar Modelo 2

In [ ]:
# Guardar el clasificador de tipos y el LabelEncoder
os.makedirs(DATA_MODELS, exist_ok=True)
joblib.dump(rf_clasificador_tipo,  MODELO_2_LGBM_PATH)
joblib.dump(label_encoder_model2,  LABEL_ENC_M2_LGBM_PATH)
print(f"Modelo 2 guardado: {MODELO_2_LGBM_PATH}")
print(f"Label Encoder guardado: {LABEL_ENC_M2_LGBM_PATH}")
